# 미국 언론사 RSS 과거 뉴스 수집 테스트

검색어 없이 **언론사가 제공하는 RSS 피드만으로** 과거 특정 기간에 나온 기사 목록을 전부 모을 수 있는지 피드별로 테스트하는 노트북입니다.
일반 RSS는 최신 기사 10~100건만 보여주기 때문에, 과거로 거슬러 가는 방법이 있는 피드만 쓸 수 있습니다.

**과거 접근 방식 (우선순위 순)**

| 방식 | 원리 | 장점 / 한계 |
|---|---|---|
| `DATE_ARCHIVE` | WordPress 날짜 아카이브 피드 `/{YYYY}/{MM}/{DD}/feed/?paged=N` | 날짜를 직접 지정, 그날 기사를 끝 페이지까지 전부. 사이트 전체 피드만 가능 |
| `PAGED` | 최신순 피드에 `?paged=N`(또는 `page=N`)을 붙여 과거 페이지로 이동 | 목표 기간이 있는 페이지를 이진탐색. 페이지 상한까지만 내려감 |

**수집에서 제외한 방식**: archive.org(Wayback Machine)에 저장된 피드 스냅샷 이어붙이기.
CNBC·WSJ·MarketWatch·NYT 등은 2025-10-02 무렵부터 15분 간격 스냅샷이 있어 이론상 가능하지만,
스냅샷 본문 요청이 Python에서는 4초 간격 + 휴식 후에도 대부분 HTTP 429로 막혀서 제외했습니다 (2026-11-04 테스트).
2-3단계에서 스냅샷 유무만 참고용으로 조회해 `WAYBACK_ONLY_FEEDS`에 남깁니다.

**단계**
1. 라이브 피드 점검 (응답, 기사 수, 날짜 파싱, 갱신 여부)
2. 과거 접근 방식 판별 + 깊이 테스트 (어디까지 과거로 내려가는지)
3. 테스트 기간 실제 수집
4. 판정 → 피드 딕셔너리 정리 (`PASSED_*`, `PARTIAL_FEEDS`, `LIVE_ONLY_FEEDS`, `WAYBACK_ONLY_FEEDS`, `FAILED_FEEDS`)
5. 실제 수집용 함수 `collect_news(start, end)`

**캐시**: 과거 날짜 아카이브 피드, Wayback CDX 조회 결과는 바뀌지 않으므로 `newsrss_test/cache/`에 저장하고 재요청하지 않습니다.
라이브 피드와 `PAGED` 페이지는 새 기사가 올라오면 내용이 밀리므로 캐시하지 않습니다.

In [1]:
import calendar
import gzip
import hashlib
import html
import json
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests

## 0. 설정

In [2]:
# ============================================================
# 테스트 기간 (미국 동부시간 날짜, 양 끝 포함)
# ============================================================
# 약 1년 전 평일 이틀. 다른 시점은 아래 DEPTH_PROBE_DATES로 확인
TEST_START = date(2025, 10, 14)
TEST_END = date(2025, 10, 15)

# 과거 어디까지 내려갈 수 있는지 확인할 날짜 (깊이 테스트)
DEPTH_PROBE_DATES = [
    date(2025, 6, 10),
    date(2024, 10, 15),
    date(2023, 10, 17),
    date(2022, 10, 18),
]

NEWS_TIMEZONE = ZoneInfo("America/New_York")

# 최신 기사가 이보다 오래됐으면 갱신이 멈춘 피드(STALE)
STALE_DAYS = 7

# 날짜가 있는 기사 비율이 이보다 낮으면 NO_DATE
MIN_DATE_RATE = 0.8

# 날짜 아카이브: 요청한 날짜(±1일)에 속한 기사 비율이 이 이상이어야 통과
MIN_ARCHIVE_MATCH_RATE = 0.8

# 무한 루프 방지용 상한
MAX_PAGES_PER_DAY = 60
MAX_PAGE_NUMBER = 50000

REQUEST_TIMEOUT = 30
REQUEST_DELAY = 0.5     # 같은 호스트 연속 요청 간격(초)
MAX_RETRIES = 3

# archive.org CDX 조회 간격(초). 429를 받으면 WAYBACK_BLOCK_WAIT × 시도 횟수만큼 쉬고 재시도
WAYBACK_DELAY = 2.0
WAYBACK_BLOCK_WAIT = 60

# 이보다 과거 날짜는 깨진 값으로 간주 (예: Fast Company 옛 기사의 1969-12-31)
MIN_VALID_DATE = datetime(1995, 1, 1, tzinfo=timezone.utc)

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36"
    ),
    "Accept": "application/rss+xml, application/atom+xml, application/xml;q=0.9, */*;q=0.8",
}

BASE_DIR = Path("newsrss_test")
CACHE_DIR = BASE_DIR / "cache"
FEED_CACHE_DIR = CACHE_DIR / "feeds"
CDX_CACHE_PATH = CACHE_DIR / "wayback_cdx.jsonl"

LIVE_RESULT_PATH = BASE_DIR / "live_check.csv"
ACCESS_RESULT_PATH = BASE_DIR / "access_check.csv"
ARTICLES_PATH = BASE_DIR / f"articles_{TEST_START}_{TEST_END}.parquet"
SUMMARY_PATH = BASE_DIR / "feed_summary.csv"
FEED_LISTS_PATH = BASE_DIR / "feed_lists.json"

In [3]:
# ============================================================
# 테스트 후보 피드
#   feed_id: {source: 도메인, name: 언론사명, section: 섹션, url: 피드 URL}
#
# - 언론사 하나가 섹션 피드를 여러 개 제공하면 섹션별로 따로 둡니다.
# - WordPress 사이트는 사이트 전체 피드(/feed/)여야 날짜 아카이브 테스트 대상이 됩니다.
# - 2026-11-04 사전 조사에서 이미 응답이 없던 피드도 기록을 위해 포함 (1단계에서 걸러짐)
# ============================================================
def feed(source, name, section, url):
    return {"source": source, "name": name, "section": section, "url": url}


CNBC = "https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id="
DJ = "https://feeds.content.dowjones.io/public/rss/"

CANDIDATE_FEEDS = {
    # ── 미국 증시/금융 전문 ──
    "cnbc_top": feed("cnbc.com", "CNBC", "Top News", CNBC + "100003114"),
    "cnbc_business": feed("cnbc.com", "CNBC", "Business", CNBC + "10001147"),
    "cnbc_investing": feed("cnbc.com", "CNBC", "Investing", CNBC + "15839069"),
    "cnbc_market_insider": feed("cnbc.com", "CNBC", "Market Insider", CNBC + "20409666"),
    "cnbc_economy": feed("cnbc.com", "CNBC", "Economy", CNBC + "20910258"),
    "cnbc_earnings": feed("cnbc.com", "CNBC", "Earnings", CNBC + "15839135"),
    "cnbc_finance": feed("cnbc.com", "CNBC", "Finance", CNBC + "10000664"),
    "cnbc_tech": feed("cnbc.com", "CNBC", "Tech", CNBC + "19854910"),

    "wsj_markets": feed("wsj.com", "Wall Street Journal", "Markets", DJ + "RSSMarketsMain"),
    "wsj_business": feed("wsj.com", "Wall Street Journal", "US Business", DJ + "WSJcomUSBusiness"),
    "wsj_economy": feed("wsj.com", "Wall Street Journal", "Economy", DJ + "socialeconomyfeed"),
    "wsj_tech": feed("wsj.com", "Wall Street Journal", "Technology", DJ + "RSSWSJD"),
    "wsj_world": feed("wsj.com", "Wall Street Journal", "World", DJ + "RSSWorldNews"),

    "marketwatch_top": feed("marketwatch.com", "MarketWatch", "Top Stories", DJ + "mw_topstories"),
    "marketwatch_bulletins": feed("marketwatch.com", "MarketWatch", "Breaking News", DJ + "mw_bulletins"),
    "marketwatch_pulse": feed("marketwatch.com", "MarketWatch", "MarketPulse", DJ + "mw_marketpulse"),

    "barrons": feed("barrons.com", "Barron's", "All", "https://www.barrons.com/feed"),
    "bloomberg_markets": feed("bloomberg.com", "Bloomberg", "Markets", "https://feeds.bloomberg.com/markets/news.rss"),
    "bloomberg_economics": feed("bloomberg.com", "Bloomberg", "Economics", "https://feeds.bloomberg.com/economics/news.rss"),
    "bloomberg_tech": feed("bloomberg.com", "Bloomberg", "Technology", "https://feeds.bloomberg.com/technology/news.rss"),

    "ibd": feed("investors.com", "Investor's Business Daily", "All", "https://www.investors.com/feed/"),
    "seekingalpha_all": feed("seekingalpha.com", "Seeking Alpha", "All Articles", "https://seekingalpha.com/feed.xml"),
    "seekingalpha_currents": feed("seekingalpha.com", "Seeking Alpha", "Market Currents", "https://seekingalpha.com/market_currents.xml"),
    "fool": feed("fool.com", "The Motley Fool", "All", "https://www.fool.com/a/feeds/foolwatch?apikey=foolwatch-feed"),
    "nasdaq_markets": feed("nasdaq.com", "Nasdaq", "Markets", "https://www.nasdaq.com/feed/rssoutbound?category=Markets"),
    "investing_news": feed("investing.com", "Investing.com", "All News", "https://www.investing.com/rss/news.rss"),
    "investing_economy": feed("investing.com", "Investing.com", "Economy", "https://www.investing.com/rss/news_14.rss"),
    "marketbeat": feed("marketbeat.com", "MarketBeat", "All", "https://www.marketbeat.com/feed/"),
    "thestreet": feed("thestreet.com", "TheStreet", "All", "https://www.thestreet.com/.rss/full/"),
    "kiplinger": feed("kiplinger.com", "Kiplinger", "All", "https://www.kiplinger.com/feeds.xml"),
    "yahoo_finance": feed("finance.yahoo.com", "Yahoo Finance", "All", "https://finance.yahoo.com/news/rssindex"),
    "zacks": feed("zacks.com", "Zacks", "All", "https://scr.zacks.com/rss"),

    # ── 경제/비즈니스 종합 ──
    "fortune": feed("fortune.com", "Fortune", "All", "https://fortune.com/feed/"),
    "business_insider": feed("businessinsider.com", "Business Insider", "All", "https://feeds.businessinsider.com/custom/all"),
    "foxbusiness_latest": feed("foxbusiness.com", "Fox Business", "Latest", "https://moxie.foxbusiness.com/google-publisher/latest.xml"),
    "foxbusiness_markets": feed("foxbusiness.com", "Fox Business", "Markets", "https://moxie.foxbusiness.com/google-publisher/markets.xml"),
    "foxbusiness_economy": feed("foxbusiness.com", "Fox Business", "Economy", "https://moxie.foxbusiness.com/google-publisher/economy.xml"),
    "cbs_moneywatch": feed("cbsnews.com", "CBS News", "MoneyWatch", "https://www.cbsnews.com/latest/rss/moneywatch"),
    "nypost": feed("nypost.com", "New York Post", "All", "https://nypost.com/feed/"),
    "nypost_business": feed("nypost.com", "New York Post", "Business", "https://nypost.com/business/feed/"),
    "usatoday_money": feed("usatoday.com", "USA Today", "Money", "https://rssfeeds.usatoday.com/UsatodaycomMoney-TopStories"),
    "cnn_money": feed("cnn.com", "CNN Business", "Money", "http://rss.cnn.com/rss/money_latest.rss"),
    "forbes_business": feed("forbes.com", "Forbes", "Business", "https://www.forbes.com/business/feed/"),
    "forbes_innovation": feed("forbes.com", "Forbes", "Innovation", "https://www.forbes.com/innovation/feed/"),
    "fastcompany": feed("fastcompany.com", "Fast Company", "Latest", "https://www.fastcompany.com/latest/rss"),
    "qz": feed("qz.com", "Quartz", "All", "https://qz.com/rss"),
    "axios": feed("axios.com", "Axios", "All", "https://api.axios.com/feed/"),
    "semafor": feed("semafor.com", "Semafor", "All", "https://www.semafor.com/rss.xml"),

    # ── 종합 일간지/방송 ──
    "nyt_business": feed("nytimes.com", "New York Times", "Business", "https://rss.nytimes.com/services/xml/rss/nyt/Business.xml"),
    "nyt_economy": feed("nytimes.com", "New York Times", "Economy", "https://rss.nytimes.com/services/xml/rss/nyt/Economy.xml"),
    "nyt_tech": feed("nytimes.com", "New York Times", "Technology", "https://rss.nytimes.com/services/xml/rss/nyt/Technology.xml"),
    "nyt_home": feed("nytimes.com", "New York Times", "Home Page", "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml"),
    "wapo_business": feed("washingtonpost.com", "Washington Post", "Business", "https://feeds.washingtonpost.com/rss/business"),
    "latimes_business": feed("latimes.com", "Los Angeles Times", "Business", "https://www.latimes.com/business/rss2.0.xml"),
    "guardian_us_business": feed("theguardian.com", "The Guardian", "US Business", "https://www.theguardian.com/us/business/rss"),
    "npr_business": feed("npr.org", "NPR", "Business", "https://feeds.npr.org/1006/rss.xml"),
    "npr_economy": feed("npr.org", "NPR", "Economy", "https://feeds.npr.org/1017/rss.xml"),
    "abc_business": feed("abcnews.go.com", "ABC News", "Business", "https://abcnews.go.com/abcnews/moneyheadlines"),
    "nbc_business": feed("nbcnews.com", "NBC News", "Business", "https://feeds.nbcnews.com/nbcnews/public/business"),
    "apnews": feed("apnews.com", "Associated Press", "All", "https://apnews.com/index.rss"),
    "yahoo_news": feed("yahoo.com", "Yahoo News", "All", "https://news.yahoo.com/rss/"),
    "politico": feed("politico.com", "Politico", "Politics", "https://rss.politico.com/politics-news.xml"),
    "thehill": feed("thehill.com", "The Hill", "All", "https://thehill.com/feed/"),
    "time": feed("time.com", "TIME", "All", "https://time.com/feed/"),
    "washington_examiner": feed("washingtonexaminer.com", "Washington Examiner", "All", "https://www.washingtonexaminer.com/feed/"),
    "nationalreview": feed("nationalreview.com", "National Review", "All", "https://www.nationalreview.com/feed/"),
    "motherjones": feed("motherjones.com", "Mother Jones", "All", "https://www.motherjones.com/feed/"),

    # ── IT/반도체 ──
    "techcrunch": feed("techcrunch.com", "TechCrunch", "All", "https://techcrunch.com/feed/"),
    "theverge": feed("theverge.com", "The Verge", "All", "https://www.theverge.com/rss/index.xml"),
    "arstechnica": feed("arstechnica.com", "Ars Technica", "All", "https://arstechnica.com/feed/"),
    "wired": feed("wired.com", "WIRED", "All", "https://www.wired.com/feed/rss"),
    "engadget": feed("engadget.com", "Engadget", "All", "https://www.engadget.com/rss.xml"),
    "cnet": feed("cnet.com", "CNET", "News", "https://www.cnet.com/rss/news/"),
    "9to5mac": feed("9to5mac.com", "9to5Mac", "All", "https://9to5mac.com/feed/"),
    "tomshardware": feed("tomshardware.com", "Tom's Hardware", "All", "https://www.tomshardware.com/feeds.xml"),
    "eetimes": feed("eetimes.com", "EE Times", "All", "https://www.eetimes.com/feed/"),
    "semiengineering": feed("semiengineering.com", "Semiconductor Engineering", "All", "https://semiengineering.com/feed/"),
    "venturebeat": feed("venturebeat.com", "VentureBeat", "All", "https://venturebeat.com/feed/"),

    # ── 보도자료 배포 ──
    "businesswire": feed("businesswire.com", "Business Wire", "All", "https://feed.businesswire.com/rss/home/?rss=G1QFDERJXkJeEFtRWA=="),
    "prnewswire": feed("prnewswire.com", "PR Newswire", "All", "https://www.prnewswire.com/rss/news-releases-list.rss"),
    "globenewswire": feed("globenewswire.com", "GlobeNewswire", "Public Companies", "https://www.globenewswire.com/RssFeed/orgclass/1/feedTitle/GlobeNewswire%20-%20News%20about%20Public%20Companies"),
}

# 테스트 전에 내용을 보고 제외한 피드와 이유
EXCLUDED_FEEDS = {
    # /feed 가 뉴스가 아니라 재테크 가이드(benzinga.com/money) 글 목록이고,
    # 1페이지 기사 시각이 1초 간격으로 일괄 갱신돼 있어 작성 시각으로 쓸 수 없음
    "benzinga": "재테크 가이드 피드 (뉴스 아님, 작성 시각 신뢰 불가)",
}

print(f"후보 피드 {len(CANDIDATE_FEEDS)}개 / 언론사 {len({f['source'] for f in CANDIDATE_FEEDS.values()})}곳")

후보 피드 81개 / 언론사 56곳


## 0-1. 공통 유틸 (요청, 캐시, 피드 파싱)

In [4]:
FEED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 캐시 폴더는 git에 올리지 않음
(CACHE_DIR / ".gitignore").write_text("*\n")


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def now_utc():
    return datetime.now(timezone.utc)


def url_hash(url):
    return hashlib.sha1(url.encode("utf-8")).hexdigest()


def et_day_bounds(start, end):
    # 동부시간 날짜 구간 [start, end] → UTC [시작, 끝)
    start_dt = datetime.combine(start, datetime.min.time(), NEWS_TIMEZONE)
    end_dt = datetime.combine(end + timedelta(days=1), datetime.min.time(), NEWS_TIMEZONE)

    return start_dt.astimezone(timezone.utc), end_dt.astimezone(timezone.utc)


def date_range(start, end):
    days = []
    current = start

    while current <= end:
        days.append(current)
        current += timedelta(days=1)

    return days


def to_feed_dict(feed_ids, extra=None):
    # feed_id 리스트 → {feed_id: 후보 정보 (+추가 정보)} (CANDIDATE_FEEDS 순서 유지)
    feed_ids = set(feed_ids)
    extra = extra or {}

    return {
        feed_id: {**info, **extra.get(feed_id, {})}
        for feed_id, info in CANDIDATE_FEEDS.items()
        if feed_id in feed_ids
    }


def make_session():
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)

    return session


SESSION = make_session()
_last_request_at = {}


def polite_get(url, delay=REQUEST_DELAY):
    # 같은 호스트에 delay초 간격을 두고 요청, 429/5xx/네트워크 오류는 재시도
    host = urlparse(url).hostname

    for attempt in range(MAX_RETRIES + 1):
        wait = _last_request_at.get(host, 0) + delay - time.time()

        if wait > 0:
            time.sleep(wait)

        try:
            # archive.org는 Accept에 RSS 형식을 지정하면 요청 제한과 상관없이 바로 429를 돌려줌
            headers = {"Accept": "*/*"} if host == "web.archive.org" else None
            response = SESSION.get(url, timeout=REQUEST_TIMEOUT, headers=headers)
            error = None

        except requests.RequestException as e:
            response = None
            error = f"{type(e).__name__}: {e}"

        finally:
            _last_request_at[host] = time.time()

        retryable = response is None or response.status_code in {429, 500, 502, 503, 504}

        if not retryable or attempt == MAX_RETRIES:
            return response, error

        if host == "web.archive.org":
            time.sleep(WAYBACK_BLOCK_WAIT * (attempt + 1))
        else:
            time.sleep(5 * (attempt + 1))


def entry_published(entry):
    # feedparser가 UTC로 정규화한 시각 (published → updated 순)
    for key in ("published_parsed", "updated_parsed"):
        value = entry.get(key)

        if value:
            published = datetime.fromtimestamp(calendar.timegm(value), timezone.utc)
            return published if published >= MIN_VALID_DATE else None

    return None


def parse_feed(content):
    parsed = feedparser.parse(content)
    entries = []

    for entry in parsed.entries:
        published_utc = entry_published(entry)

        entries.append({
            # Atom 피드(The Hill 등)는 제목에 &#8216; 같은 엔티티가 남아 있어 한 번 더 해제
            "title": html.unescape(entry.get("title") or "").strip(),
            "link": entry.get("link"),
            "guid": entry.get("id") or entry.get("link"),
            "published_utc": published_utc,
            "published_et": published_utc.astimezone(NEWS_TIMEZONE) if published_utc else None,
            "summary": html.unescape(entry.get("summary") or "").strip(),
            "author": entry.get("author"),
            "tags": [t.get("term") for t in entry.get("tags", []) if t.get("term")],
        })

    return entries


def fetch_feed(url, use_cache=False, delay=REQUEST_DELAY):
    '''
    피드 요청 → {"url", "http_status", "error", "entries", "from_cache"}
    use_cache=True면 200/404 응답을 저장해두고 재요청하지 않음 (과거 날짜 아카이브 전용)
    '''
    path_ok = FEED_CACHE_DIR / f"{url_hash(url)}.xml.gz"
    path_404 = FEED_CACHE_DIR / f"{url_hash(url)}.404"

    if use_cache and path_ok.exists():
        content = gzip.decompress(path_ok.read_bytes())
        return {"url": url, "http_status": 200, "error": None, "entries": parse_feed(content), "from_cache": True}

    if use_cache and path_404.exists():
        return {"url": url, "http_status": 404, "error": None, "entries": [], "from_cache": True}

    response, error = polite_get(url, delay=delay)

    if response is None:
        return {"url": url, "http_status": None, "error": error, "entries": [], "from_cache": False}

    entries = parse_feed(response.content) if response.status_code == 200 else []

    if use_cache and response.status_code == 200:
        path_ok.write_bytes(gzip.compress(response.content))

    if use_cache and response.status_code == 404:
        path_404.touch()

    return {"url": url, "http_status": response.status_code, "error": None, "entries": entries, "from_cache": False}


def add_param(url, key, value):
    return f"{url}{'&' if '?' in url else '?'}{key}={value}"


def entry_times(entries):
    return [e["published_utc"] for e in entries if e["published_utc"]]

## 1. 라이브 피드 점검

현재 피드가 응답하는지, 기사에 날짜가 있는지, 갱신이 멈추지 않았는지 확인합니다.
`span_hours`는 피드 하나가 담고 있는 기간입니다 (최신 기사 ~ 가장 오래된 기사).

In [5]:
def check_live(feeds):
    rows = []

    for feed_id, info in feeds.items():
        result = fetch_feed(info["url"])
        entries = result["entries"]
        times = entry_times(entries)

        if result["http_status"] != 200:
            status = "HTTP_ERROR"
        elif not entries:
            status = "EMPTY"
        elif len(times) / len(entries) < MIN_DATE_RATE:
            status = "NO_DATE"
        elif max(times) < now_utc() - timedelta(days=STALE_DAYS):
            status = "STALE"
        else:
            status = "OK"

        rows.append({
            "feed_id": feed_id,
            "source": info["source"],
            "name": info["name"],
            "section": info["section"],
            "live_status": status,
            "http_status": result["http_status"],
            "entries": len(entries),
            "date_rate": round(len(times) / len(entries), 2) if entries else None,
            "newest_et": max(times).astimezone(NEWS_TIMEZONE) if times else None,
            "oldest_et": min(times).astimezone(NEWS_TIMEZONE) if times else None,
            "span_hours": round((max(times) - min(times)).total_seconds() / 3600, 1) if times else None,
            "error": result["error"],
        })

    return pd.DataFrame(rows)


live_df = check_live(CANDIDATE_FEEDS)
live_df.to_csv(LIVE_RESULT_PATH, index=False, encoding="utf-8-sig")

# ── 변수 정리: 라이브 점검 ──
LIVE_OK_FEEDS = to_feed_dict(live_df.loc[live_df["live_status"] == "OK", "feed_id"])
LIVE_FAILED_FEEDS = (
    live_df.loc[live_df["live_status"] != "OK"]
    .set_index("feed_id")["live_status"]
    .to_dict()
)

print(f"라이브 정상 {len(LIVE_OK_FEEDS)}개 / 실패 {len(LIVE_FAILED_FEEDS)}개")
print("실패:", LIVE_FAILED_FEEDS)

live_df[["feed_id", "name", "section", "live_status", "http_status", "entries", "date_rate", "newest_et", "span_hours"]]

라이브 정상 71개 / 실패 10개
실패: {'marketwatch_pulse': 'STALE', 'barrons': 'HTTP_ERROR', 'yahoo_finance': 'HTTP_ERROR', 'zacks': 'HTTP_ERROR', 'usatoday_money': 'HTTP_ERROR', 'cnn_money': 'STALE', 'qz': 'EMPTY', 'wapo_business': 'HTTP_ERROR', 'apnews': 'HTTP_ERROR', 'globenewswire': 'HTTP_ERROR'}


,feed_id,name,section,live_status,http_status,entries,date_rate,newest_et,span_hours
0,cnbc_top,CNBC,Top News,OK,200,30,1.0,2026-11-03 23:08:52-05:00,31.0
1,cnbc_business,CNBC,Business,OK,200,30,1.0,2026-11-03 17:26:20-05:00,176.3
2,cnbc_investing,CNBC,Investing,OK,200,30,1.0,2026-11-03 19:12:42-05:00,122.6
3,cnbc_market_insider,CNBC,Market Insider,OK,200,30,1.0,2026-11-03 18:30:34-05:00,342.8
4,cnbc_economy,CNBC,Economy,OK,200,30,1.0,2026-11-03 14:15:21-05:00,485.4
...,...,...,...,...,...,...,...,...,...
76,semiengineering,Semiconductor Engineering,All,OK,200,10,1.0,2026-11-03 13:01:10-05:00,34.0
77,venturebeat,VentureBeat,All,OK,200,7,1.0,2026-11-02 17:16:00-05:00,103.8
78,businesswire,Business Wire,All,OK,200,36,1.0,2026-11-03 20:45:00-05:00,17.1
79,prnewswire,PR Newswire,All,OK,200,20,1.0,2026-11-03 22:51:00-05:00,1.9


## 2. 과거 접근 방식 판별 + 깊이 테스트

### 2-1. `DATE_ARCHIVE`: WordPress 날짜 아카이브 피드

사이트 전체 피드(`/feed/`)인 후보만 대상입니다. RSS(`/feed/`)가 안 되면 Atom(`/feed/atom/`)도 시도합니다.
(The Hill은 날짜 아카이브 RSS의 1페이지만 비어 있고 Atom은 정상인 사례)

In [6]:
ARCHIVE_FORMATS = {
    "rss": "{root}/{day:%Y/%m/%d}/feed/",
    "atom": "{root}/{day:%Y/%m/%d}/feed/atom/",
}


def site_root(url):
    parsed = urlparse(url)

    return f"{parsed.scheme}://{parsed.netloc}"


def is_site_wide_wp_feed(url):
    return urlparse(url).path.rstrip("/") == "/feed"


def archive_url(info, day, page=1, fmt="rss"):
    url = ARCHIVE_FORMATS[fmt].format(root=site_root(info["url"]), day=day)

    return url if page == 1 else add_param(url, "paged", page)


def archive_match_rate(entries, day):
    # 아카이브 기사 중 요청 날짜(동부시간 ±1일)에 속하는 비율, 정확히 그 날짜인 비율
    days = [t.astimezone(NEWS_TIMEZONE).date() for t in entry_times(entries)]

    if not days:
        return 0.0, 0.0

    near = sum(abs((d - day).days) <= 1 for d in days) / len(days)
    exact = sum(d == day for d in days) / len(days)

    return near, exact


def check_date_archive(feeds, test_day, depth_days):
    rows = []

    for feed_id, info in feeds.items():
        if not is_site_wide_wp_feed(info["url"]):
            continue

        row = {"feed_id": feed_id, "archive_ok": False, "archive_format": None}

        for fmt in ARCHIVE_FORMATS:
            result = fetch_feed(archive_url(info, test_day, fmt=fmt), use_cache=True)
            near, exact = archive_match_rate(result["entries"], test_day)

            row.update({
                "archive_http": result["http_status"],
                "archive_entries": len(result["entries"]),
                "archive_near_rate": round(near, 2),
                "archive_exact_rate": round(exact, 2),
            })

            if result["entries"] and near >= MIN_ARCHIVE_MATCH_RATE:
                row.update({"archive_ok": True, "archive_format": fmt})
                break

        # 깊이 테스트: 과거 날짜의 아카이브 1페이지가 나오는지
        if row["archive_ok"]:
            for day in depth_days:
                result = fetch_feed(archive_url(info, day, fmt=row["archive_format"]), use_cache=True)
                near, _ = archive_match_rate(result["entries"], day)
                row[f"depth_{day}"] = bool(result["entries"]) and near >= MIN_ARCHIVE_MATCH_RATE

        rows.append(row)

    return pd.DataFrame(rows)


archive_df = check_date_archive(CANDIDATE_FEEDS, TEST_START, DEPTH_PROBE_DATES)
archive_df

,feed_id,archive_ok,archive_format,archive_http,archive_entries,archive_near_rate,archive_exact_rate,depth_2025-06-10,depth_2024-10-15,depth_2023-10-17,depth_2022-10-18
0,barrons,False,None,403.0,0,0.0,0.0,NaN,NaN,NaN,NaN
1,ibd,True,rss,200.0,63,1.0,1.0,True,True,True,True
2,marketbeat,False,None,404.0,0,0.0,0.0,NaN,NaN,NaN,NaN
3,fortune,False,None,404.0,0,0.0,0.0,NaN,NaN,NaN,NaN
4,nypost,True,rss,200.0,20,1.0,1.0,True,True,True,True
5,axios,False,None,404.0,0,0.0,0.0,NaN,NaN,NaN,NaN
6,thehill,True,atom,200.0,15,1.0,1.0,True,True,True,True
7,time,False,None,406.0,0,0.0,0.0,NaN,NaN,NaN,NaN
8,washington_examiner,False,None,404.0,0,0.0,0.0,NaN,NaN,NaN,NaN
9,nationalreview,True,rss,200.0,19,1.0,1.0,True,True,True,True


### 2-2. `PAGED`: 최신순 피드 페이지 이동

날짜 아카이브가 안 되는 라이브 정상 피드만 대상입니다.
`paged=2` → `page=2` 순서로 시도해서 2페이지가 1페이지와 다르고 더 과거 기사면 통과,
이후 페이지 번호를 2배씩 늘리고 이진탐색으로 **마지막 페이지와 가장 오래된 기사 날짜**를 찾습니다.

In [7]:
PAGE_PARAMS = ["paged", "page"]


def paged_url(info, param, page):
    return info["url"] if page == 1 else add_param(info["url"], param, page)


def page_oldest(info, param, page, memo):
    # 페이지의 가장 오래된 기사 시각 (빈 페이지/오류면 None)
    if page not in memo:
        result = fetch_feed(paged_url(info, param, page))
        times = entry_times(result["entries"])
        memo[page] = (min(times), max(times), result["entries"]) if times else None

    return memo[page]


def find_last_page(info, param, memo):
    '''
    1, 2, 4, 8 … 로 비는 페이지를 찾고 그 사이를 이진탐색.
    파라미터를 무시하고 같은 내용을 주는 사이트가 있어서
    앞 페이지보다 더 과거 기사가 있어야 유효한 페이지로 봄
    '''
    def is_deeper(page, than):
        current = page_oldest(info, param, page, memo)
        return current is not None and current[0] < page_oldest(info, param, than, memo)[0]

    good, bad = 1, None
    page = 2

    while page <= MAX_PAGE_NUMBER:
        if is_deeper(page, good):
            good = page
            page *= 2
        else:
            bad = page
            break

    if bad is None:
        return good

    while bad - good > 1:
        mid = (good + bad) // 2

        if is_deeper(mid, good):
            good = mid
        else:
            bad = mid

    return good


def check_paged(feeds):
    rows = []

    for feed_id, info in feeds.items():
        row = {"feed_id": feed_id, "paged_ok": False, "page_param": None}
        first = fetch_feed(info["url"])
        first_links = {e["link"] for e in first["entries"]}
        first_times = entry_times(first["entries"])

        for param in PAGE_PARAMS:
            memo = {}
            second = page_oldest(info, param, 2, memo)

            if not second or not first_times:
                continue

            second_links = {e["link"] for e in second[2]}

            # 2페이지가 1페이지와 다르고, 2페이지의 최신 기사가 1페이지 중간보다 과거여야 함
            is_older = second[1] <= sorted(first_times)[len(first_times) // 2]

            if second_links != first_links and is_older:
                last_page = find_last_page(info, param, memo)
                oldest = page_oldest(info, param, last_page, memo)[0]

                row.update({
                    "paged_ok": True,
                    "page_param": param,
                    "page_size": len(first["entries"]),
                    "last_page": last_page,
                    "paged_oldest_et": oldest.astimezone(NEWS_TIMEZONE),
                })
                break

        rows.append(row)

    return pd.DataFrame(rows)


archive_ok_ids = set(archive_df.loc[archive_df["archive_ok"], "feed_id"])

paged_df = check_paged({
    feed_id: info
    for feed_id, info in LIVE_OK_FEEDS.items()
    if feed_id not in archive_ok_ids
})

paged_df[paged_df["paged_ok"]]

,feed_id,paged_ok,page_param,page_size,last_page,paged_oldest_et
1,cnbc_business,True,page,30.0,7.0,2026-09-10 11:39:23-04:00
2,cnbc_investing,True,page,30.0,7.0,2026-10-07 07:38:42-04:00
3,cnbc_market_insider,True,page,30.0,7.0,2026-07-09 07:59:47-04:00
4,cnbc_economy,True,page,30.0,7.0,2026-05-18 12:08:35-04:00
6,cnbc_finance,True,page,30.0,7.0,2026-09-04 09:52:56-04:00
7,cnbc_tech,True,page,30.0,7.0,2026-10-15 08:15:09-04:00
33,nypost_business,True,paged,20.0,10.0,2026-10-20 18:04:48-04:00
36,fastcompany,True,page,19.0,10530.0,1995-01-01 05:00:00-05:00
54,wired,True,page,50.0,200.0,2024-06-06 12:00:00-04:00
58,eetimes,True,paged,10.0,24759.0,1995-07-01 14:23:00-04:00


### 2-3. (참고용) archive.org 피드 스냅샷 유무

**수집에는 쓰지 않습니다** (맨 위 '수집에서 제외한 방식' 참고). 과거 접근 방법이 없는 피드 중
Wayback 스냅샷이라도 남아 있는 피드를 기록해두기 위한 조회입니다.
CDX API로 테스트 기간(앞뒤 하루 포함)의 스냅샷 수와 최대 간격을 보고, 스냅샷이 있으면 깊이 테스트 날짜별 스냅샷 수도 셉니다.

In [8]:
CDX_URL = "https://web.archive.org/cdx/search/cdx"


def wayback_snapshots(feed_url, start_utc, end_utc):
    # [start_utc, end_utc] 사이 스냅샷 [(capture_utc, original_url)] (과거 구간만 캐시)
    key = f"{feed_url}|{start_utc:%Y%m%d%H%M%S}|{end_utc:%Y%m%d%H%M%S}"

    for record in load_jsonl(CDX_CACHE_PATH):
        if record["key"] == key:
            return [(datetime.fromisoformat(t), u) for t, u in record["snapshots"]]

    params = (
        f"?url={requests.utils.quote(feed_url, safe='')}"
        f"&from={start_utc:%Y%m%d%H%M%S}&to={end_utc:%Y%m%d%H%M%S}"
        "&output=json&filter=statuscode:200&collapse=digest&fl=timestamp,original"
    )
    response, error = polite_get(CDX_URL + params, delay=WAYBACK_DELAY)

    if response is None or response.status_code != 200:
        raise RuntimeError(f"CDX 실패: {feed_url} {error or response.status_code}")

    try:
        rows = response.json()[1:] if response.text.strip() else []
    except ValueError:
        raise RuntimeError(f"CDX 응답 해석 실패: {feed_url}")
    snapshots = [
        (datetime.strptime(ts, "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc), original)
        for ts, original in rows
    ]

    if end_utc < now_utc() - timedelta(days=2):
        append_jsonl(CDX_CACHE_PATH, {
            "key": key,
            "snapshots": [(t.isoformat(), u) for t, u in snapshots],
        })

    return snapshots


def check_wayback(feeds, start, end, depth_days):
    window_start, window_end = et_day_bounds(start, end)
    rows = []

    for feed_id, info in feeds.items():
        row = {"feed_id": feed_id}

        try:
            snapshots = wayback_snapshots(
                info["url"],
                window_start - timedelta(days=1),
                window_end + timedelta(days=1),
            )

        except RuntimeError as e:
            rows.append({**row, "wayback_error": str(e)})
            continue

        in_window = [t for t, _ in snapshots if window_start <= t <= window_end]
        edges = [window_start] + in_window + [window_end]
        gaps = [(b - a).total_seconds() / 3600 for a, b in zip(edges, edges[1:])]

        row.update({
            "wayback_snapshots": len(in_window),
            "wayback_max_gap_hours": round(max(gaps), 1),
        })

        if in_window:
            for day in depth_days:
                day_start, day_end = et_day_bounds(day, day)
                try:
                    row[f"wb_depth_{day}"] = len(wayback_snapshots(info["url"], day_start, day_end))
                except RuntimeError:
                    row[f"wb_depth_{day}"] = None

        rows.append(row)

    return pd.DataFrame(rows)


wayback_df = check_wayback(CANDIDATE_FEEDS, TEST_START, TEST_END, DEPTH_PROBE_DATES)
wayback_df[wayback_df["wayback_snapshots"].fillna(0) > 0]

,feed_id,wayback_snapshots,wayback_max_gap_hours,wb_depth_2025-06-10,wb_depth_2024-10-15,wb_depth_2023-10-17,wb_depth_2022-10-18,wayback_error
0,cnbc_top,132.0,2.3,0.0,0.0,0.0,1.0,NaN
8,wsj_markets,190.0,1.6,0.0,0.0,0.0,0.0,NaN
9,wsj_business,186.0,1.0,0.0,0.0,0.0,0.0,NaN
10,wsj_economy,189.0,0.8,0.0,0.0,0.0,0.0,NaN
11,wsj_tech,188.0,1.6,0.0,0.0,0.0,0.0,NaN
12,wsj_world,189.0,0.9,0.0,0.0,0.0,0.0,NaN
13,marketwatch_top,193.0,2.4,0.0,0.0,0.0,0.0,NaN
18,bloomberg_economics,183.0,1.0,0.0,0.0,0.0,0.0,NaN
21,seekingalpha_all,152.0,1.1,0.0,0.0,0.0,0.0,NaN
25,investing_news,154.0,1.2,0.0,0.0,0.0,0.0,NaN


### 2-4. 피드별 수집 방식 결정

우선순위: `DATE_ARCHIVE` → `PAGED`(테스트 시작일까지 내려가는 경우).
둘 다 안 되지만 테스트 기간 Wayback 스냅샷은 있는 피드는 `WAYBACK_ONLY_FEEDS`로 따로 기록합니다.

In [9]:
access_df = (
    live_df[["feed_id", "source", "name", "section", "live_status", "entries", "span_hours"]]
    .merge(archive_df, on="feed_id", how="left")
    .merge(paged_df, on="feed_id", how="left")
    .merge(wayback_df, on="feed_id", how="left")
)

test_start_utc, _ = et_day_bounds(TEST_START, TEST_END)

access_df["paged_reaches_test"] = (
    access_df["paged_oldest_et"].notna()
    & (pd.to_datetime(access_df["paged_oldest_et"], utc=True) <= test_start_utc)
)


def choose_method(row):
    if row["archive_ok"] == True:
        return "DATE_ARCHIVE"

    if row["paged_reaches_test"]:
        return "PAGED"

    return None


access_df["method"] = access_df.apply(choose_method, axis=1)
access_df.to_csv(ACCESS_RESULT_PATH, index=False, encoding="utf-8-sig")

# ── 변수 정리: 접근 방식 ──
DATE_ARCHIVE_FEEDS = to_feed_dict(access_df.loc[access_df["method"] == "DATE_ARCHIVE", "feed_id"])
PAGED_FEEDS = to_feed_dict(access_df.loc[access_df["method"] == "PAGED", "feed_id"])
NO_PAST_ACCESS_FEEDS = to_feed_dict(access_df.loc[access_df["method"].isna(), "feed_id"])

# 과거 접근은 안 되지만 테스트 기간 Wayback 스냅샷은 있는 피드 (참고용, 수집 안 함)
WAYBACK_ONLY_FEEDS = to_feed_dict(
    access_df.loc[access_df["method"].isna() & (access_df["wayback_snapshots"].fillna(0) > 0), "feed_id"]
)

for label, feeds in [
    ("DATE_ARCHIVE", DATE_ARCHIVE_FEEDS),
    ("PAGED", PAGED_FEEDS),
    ("과거 접근 불가", NO_PAST_ACCESS_FEEDS),
    ("└ Wayback만 있음", WAYBACK_ONLY_FEEDS),
]:
    print(f"{label:14s} {len(feeds):2d}개: {list(feeds)}")

depth_cols = [c for c in access_df.columns if c.startswith("depth_")]

access_df.loc[
    access_df["method"].notna(),
    ["feed_id", "name", "section", "method", "archive_format", "page_param", "page_size",
     "last_page", "paged_oldest_et"] + depth_cols,
]

DATE_ARCHIVE    9개: ['ibd', 'nypost', 'thehill', 'nationalreview', 'motherjones', 'techcrunch', 'arstechnica', '9to5mac', 'semiengineering']
PAGED           3개: ['fastcompany', 'wired', 'eetimes']
과거 접근 불가       69개: ['cnbc_top', 'cnbc_business', 'cnbc_investing', 'cnbc_market_insider', 'cnbc_economy', 'cnbc_earnings', 'cnbc_finance', 'cnbc_tech', 'wsj_markets', 'wsj_business', 'wsj_economy', 'wsj_tech', 'wsj_world', 'marketwatch_top', 'marketwatch_bulletins', 'marketwatch_pulse', 'barrons', 'bloomberg_markets', 'bloomberg_economics', 'bloomberg_tech', 'seekingalpha_all', 'seekingalpha_currents', 'fool', 'nasdaq_markets', 'investing_news', 'investing_economy', 'marketbeat', 'thestreet', 'kiplinger', 'yahoo_finance', 'zacks', 'fortune', 'business_insider', 'foxbusiness_latest', 'foxbusiness_markets', 'foxbusiness_economy', 'cbs_moneywatch', 'nypost_business', 'usatoday_money', 'cnn_money', 'forbes_business', 'forbes_innovation', 'qz', 'axios', 'semafor', 'nyt_business', 'nyt_economy', '

,feed_id,name,section,method,archive_format,page_param,page_size,last_page,paged_oldest_et,depth_2025-06-10,depth_2024-10-15,depth_2023-10-17,depth_2022-10-18
20,ibd,Investor's Business Daily,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
38,nypost,New York Post,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
44,fastcompany,Fast Company,Latest,PAGED,NaN,page,19.0,10530.0,1995-01-01 05:00:00-05:00,NaN,NaN,NaN,NaN
62,thehill,The Hill,All,DATE_ARCHIVE,atom,NaN,NaN,NaN,NaT,True,True,True,True
65,nationalreview,National Review,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
66,motherjones,Mother Jones,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
67,techcrunch,TechCrunch,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
69,arstechnica,Ars Technica,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True
70,wired,WIRED,All,PAGED,NaN,page,50.0,200.0,2024-06-06 12:00:00-04:00,NaN,NaN,NaN,NaN
73,9to5mac,9to5Mac,All,DATE_ARCHIVE,rss,NaN,NaN,NaN,NaT,True,True,True,True


## 3. 테스트 기간 실제 수집

2단계에서 정한 방식으로 테스트 기간의 기사를 실제로 모읍니다. 두 수집 함수는 5단계 실사용 함수에서도 그대로 씁니다.

- `DATE_ARCHIVE`: 사이트 시간대가 동부시간이 아닐 수 있어 앞뒤 하루 아카이브까지 받은 뒤 동부시간 날짜로 거름.
  날짜마다 빈 페이지(404)가 나올 때까지 받았으면 `complete=True`
- `PAGED`: 기간 끝이 들어있는 페이지를 이진탐색으로 찾고 기간 시작보다 과거가 나올 때까지 다음 페이지로 진행.
  기간 시작보다 과거 기사가 나오기 전에 페이지가 끝나면 `complete=False`

In [10]:
def finalize_articles(feed_id, entries, start, end, method):
    # 기간 필터 + 링크 기준 중복 제거
    window_start, window_end = et_day_bounds(start, end)
    info = CANDIDATE_FEEDS[feed_id]
    seen = {}

    for entry in entries:
        published = entry["published_utc"]

        if published is None or not (window_start <= published < window_end):
            continue

        key = entry["link"] or entry["guid"]

        if key not in seen or published < seen[key]["published_utc"]:
            seen[key] = entry

    return [
        {
            "feed_id": feed_id,
            "source": info["source"],
            "name": info["name"],
            "section": info["section"],
            "method": method,
            **entry,
        }
        for entry in seen.values()
    ]


def collect_date_archive(feed_id, info, start, end, fmt="rss"):
    entries = []
    complete = True
    requests_made = 0

    for day in date_range(start - timedelta(days=1), end + timedelta(days=1)):
        seen_links = set()

        for page in range(1, MAX_PAGES_PER_DAY + 1):
            result = fetch_feed(archive_url(info, day, page, fmt), use_cache=day < date.today() - timedelta(days=1))
            requests_made += 1
            links = {e["link"] for e in result["entries"]}

            # 마지막 페이지 다음은 404 또는 빈 피드, 같은 페이지 반복도 종료로 봄
            if not result["entries"] or links <= seen_links:
                if result["http_status"] not in {200, 404}:
                    complete = False
                break

            seen_links |= links
            entries.extend(result["entries"])

        else:
            complete = False

    return entries, {"complete": complete, "requests": requests_made}


def collect_paged(feed_id, info, start, end, param="paged"):
    window_start, window_end = et_day_bounds(start, end)
    memo = {}

    # 기간 끝보다 과거 기사가 처음 나오는 페이지를 찾음 (2배씩 늘린 뒤 이진탐색)
    def reaches(page):
        oldest = page_oldest(info, param, page, memo)
        return oldest is None or oldest[0] < window_end

    lo, hi = 1, 1

    while not reaches(hi):
        lo, hi = hi, min(hi * 2, MAX_PAGE_NUMBER)

        if hi == lo:
            return [], {"complete": False, "requests": len(memo)}

    while hi - lo > 1:
        mid = (lo + hi) // 2

        if reaches(mid):
            hi = mid
        else:
            lo = mid

    # 페이지 밀림 대비 한 페이지 앞부터 기간 시작보다 과거가 나올 때까지
    entries = []
    page = max(1, hi - 1)
    complete = False

    while page <= MAX_PAGE_NUMBER:
        result = page_oldest(info, param, page, memo)

        if result is None:
            break

        entries.extend(result[2])

        if result[0] < window_start:
            complete = True
            break

        page += 1

    return entries, {"complete": complete, "requests": len(memo)}


def collect_feed(feed_id, info, start, end):
    method = info["method"]

    if method == "DATE_ARCHIVE":
        entries, stats = collect_date_archive(feed_id, info, start, end, fmt=info.get("archive_format") or "rss")
    elif method == "PAGED":
        entries, stats = collect_paged(feed_id, info, start, end, param=info.get("page_param") or "paged")
    else:
        raise ValueError(f"알 수 없는 방식: {method}")

    return finalize_articles(feed_id, entries, start, end, method), stats

In [11]:
METHOD_PARAMS = (
    access_df.loc[access_df["method"].notna()]
    .set_index("feed_id")[["method", "archive_format", "page_param"]]
    .astype(object)
    .where(lambda d: d.notna(), None)
    .to_dict(orient="index")
)

TEST_TARGET_FEEDS = to_feed_dict(METHOD_PARAMS, extra=METHOD_PARAMS)

collect_rows = []
all_articles = []

for feed_id, info in TEST_TARGET_FEEDS.items():
    started = time.time()

    try:
        articles, stats = collect_feed(feed_id, info, TEST_START, TEST_END)
        error = None

    except Exception as e:
        articles, stats = [], {"complete": False}
        error = f"{type(e).__name__}: {e}"

    all_articles.extend(articles)
    collect_rows.append({
        "feed_id": feed_id,
        "method": info["method"],
        "articles": len(articles),
        **stats,
        "seconds": round(time.time() - started, 1),
        "error": error,
    })
    print(f"{feed_id:22s} {info['method']:12s} {len(articles):5d}건  {stats}  {error or ''}", flush=True)

collect_df = pd.DataFrame(collect_rows)
articles_df = pd.DataFrame(all_articles)
articles_df.to_parquet(ARTICLES_PATH, index=False)

print(f"\n저장: {ARTICLES_PATH} ({len(articles_df)}건)")

ibd                    DATE_ARCHIVE   131건  {'complete': True, 'requests': 8}  


nypost                 DATE_ARCHIVE   505건  {'complete': True, 'requests': 56}  


fastcompany            PAGED          107건  {'complete': True, 'requests': 23}  


thehill                DATE_ARCHIVE   290건  {'complete': True, 'requests': 41}  


nationalreview         DATE_ARCHIVE    47건  {'complete': True, 'requests': 11}  


motherjones            DATE_ARCHIVE    15건  {'complete': True, 'requests': 8}  


... (3 outputs left out)

9to5mac                DATE_ARCHIVE    59건  {'complete': True, 'requests': 8}  


eetimes                PAGED            9건  {'complete': True, 'requests': 14}  


semiengineering        DATE_ARCHIVE    23건  {'complete': True, 'requests': 9}  



저장: newsrss_test/articles_2025-10-14_2025-10-15.parquet (1294건)


## 4. 판정 + 피드 딕셔너리 정리

- `PASSED`: 기사가 있고 끝까지 순회함 (날짜별 마지막 페이지까지 / 기간 시작보다 과거 페이지까지)
- `PARTIAL`: 기사는 있지만 순회가 중간에 끊김 (페이지 상한, 요청 오류)
- `LIVE_ONLY`: 지금 피드는 정상이지만 과거로 갈 방법이 없음 → 앞으로 주기적으로 받아 쌓는 용도로만 사용 가능

`WAYBACK_ONLY_FEEDS`는 판정과 별개인 참고 목록입니다. `LIVE_ONLY`·`FAILED` 중 테스트 기간 Wayback 스냅샷이 있는 피드가 들어갑니다.
- `FAILED`: 라이브도 실패하고 과거 접근도 불가, 또는 수집 결과 0건

In [12]:
# 시간대별 공백 확인용: 테스트 기간 중 기사가 0건인 시간(동부시간) 수
def empty_hours(group):
    window_start, window_end = et_day_bounds(TEST_START, TEST_END)
    hours = pd.date_range(window_start, window_end, freq="h", inclusive="left")
    filled = set(pd.to_datetime(group["published_utc"], utc=True).dt.floor("h"))

    return sum(h not in filled for h in hours)


per_day = (
    articles_df
    .assign(day=lambda d: d["published_et"].map(lambda t: t.date().isoformat()))
    .pivot_table(index="feed_id", columns="day", values="link", aggfunc="count", fill_value=0)
)

summary_df = (
    access_df[["feed_id", "source", "name", "section", "live_status", "method"]]
    .merge(collect_df.drop(columns="method"), on="feed_id", how="left")
    .merge(per_day, left_on="feed_id", right_index=True, how="left")
)

summary_df["empty_hours"] = summary_df["feed_id"].map(
    articles_df.groupby("feed_id").apply(empty_hours, include_groups=False)
)


def verdict(row):
    if row["method"] is None or pd.isna(row["method"]):
        return "LIVE_ONLY" if row["live_status"] == "OK" else "FAILED"

    if not row["articles"] or pd.isna(row["articles"]):
        return "FAILED"

    return "PASSED" if row["complete"] == True else "PARTIAL"


summary_df["verdict"] = summary_df.apply(verdict, axis=1)
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

day_cols = list(per_day.columns)

summary_df[
    ["feed_id", "name", "section", "method", "verdict", "articles"] + day_cols
    + ["empty_hours", "complete", "requests", "seconds", "error"]
].sort_values(["verdict", "method", "articles"], ascending=[False, True, False])

,feed_id,name,section,method,verdict,articles,2025-10-14,2025-10-15,empty_hours,complete,requests,seconds,error
38,nypost,New York Post,All,DATE_ARCHIVE,PASSED,505.0,246.0,259.0,2.0,True,56.0,0.5,None
62,thehill,The Hill,All,DATE_ARCHIVE,PASSED,290.0,156.0,134.0,6.0,True,41.0,0.2,None
20,ibd,Investor's Business Daily,All,DATE_ARCHIVE,PASSED,131.0,63.0,68.0,25.0,True,8.0,0.1,None
73,9to5mac,9to5Mac,All,DATE_ARCHIVE,PASSED,59.0,27.0,32.0,19.0,True,8.0,0.1,None
67,techcrunch,TechCrunch,All,DATE_ARCHIVE,PASSED,55.0,28.0,27.0,21.0,True,11.0,0.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
41,cnn_money,CNN Business,Money,None,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,qz,Quartz,All,None,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,wapo_business,Washington Post,Business,None,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,apnews,Associated Press,All,None,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# ── 변수 정리: 최종 판정 ──
def feeds_where(mask, with_method=True):
    ids = summary_df.loc[mask, "feed_id"]

    return to_feed_dict(ids, extra=METHOD_PARAMS if with_method else None)


PASSED_FEEDS = feeds_where(summary_df["verdict"] == "PASSED")
PASSED_DATE_ARCHIVE_FEEDS = {k: v for k, v in PASSED_FEEDS.items() if v["method"] == "DATE_ARCHIVE"}
PASSED_PAGED_FEEDS = {k: v for k, v in PASSED_FEEDS.items() if v["method"] == "PAGED"}

PARTIAL_FEEDS = feeds_where(summary_df["verdict"] == "PARTIAL")
LIVE_ONLY_FEEDS = feeds_where(summary_df["verdict"] == "LIVE_ONLY", with_method=False)

# 실패 사유 {feed_id: 사유}
FAILED_FEEDS = {
    row["feed_id"]: (
        f"수집 0건 ({row['method']})" if pd.notna(row["method"])
        else f"라이브 {row['live_status']}, 과거 접근 불가"
    )
    for _, row in summary_df[summary_df["verdict"] == "FAILED"].iterrows()
}

# 과거로 내려갈 수 있는 깊이 (통과/부분 피드)
def depth_of(feed_id):
    row = access_df.set_index("feed_id").loc[feed_id]
    method = row["method"]

    if method == "DATE_ARCHIVE":
        ok = [str(d) for d in DEPTH_PROBE_DATES if row.get(f"depth_{d}") == True]
        return f"아카이브 확인 날짜: {', '.join(ok) or '없음'}"

    return f"마지막 페이지 {int(row['last_page'])} → {row['paged_oldest_et']:%Y-%m-%d}까지"


FEED_DEPTH = {feed_id: depth_of(feed_id) for feed_id in {**PASSED_FEEDS, **PARTIAL_FEEDS}}

feed_lists = {
    "created_at": now_utc().isoformat(),
    "test_period": [TEST_START.isoformat(), TEST_END.isoformat()],
    "depth_probe_dates": [d.isoformat() for d in DEPTH_PROBE_DATES],
    "PASSED_DATE_ARCHIVE": PASSED_DATE_ARCHIVE_FEEDS,
    "PASSED_PAGED": PASSED_PAGED_FEEDS,
    "PARTIAL": PARTIAL_FEEDS,
    "LIVE_ONLY": LIVE_ONLY_FEEDS,
    "WAYBACK_ONLY": WAYBACK_ONLY_FEEDS,
    "FAILED": FAILED_FEEDS,
    "DEPTH": FEED_DEPTH,
}

FEED_LISTS_PATH.write_text(json.dumps(feed_lists, ensure_ascii=False, indent=2))

for key in ["PASSED_DATE_ARCHIVE", "PASSED_PAGED", "PARTIAL", "LIVE_ONLY", "WAYBACK_ONLY", "FAILED"]:
    print(f"{key:20s} {len(feed_lists[key]):2d}개: {list(feed_lists[key])}")

print()

for feed_id, depth in FEED_DEPTH.items():
    print(f"{feed_id:22s} {depth}")

PASSED_DATE_ARCHIVE   9개: ['ibd', 'nypost', 'thehill', 'nationalreview', 'motherjones', 'techcrunch', 'arstechnica', '9to5mac', 'semiengineering']
PASSED_PAGED          3개: ['fastcompany', 'wired', 'eetimes']
PARTIAL               0개: []
LIVE_ONLY            59개: ['cnbc_top', 'cnbc_business', 'cnbc_investing', 'cnbc_market_insider', 'cnbc_economy', 'cnbc_earnings', 'cnbc_finance', 'cnbc_tech', 'wsj_markets', 'wsj_business', 'wsj_economy', 'wsj_tech', 'wsj_world', 'marketwatch_top', 'marketwatch_bulletins', 'bloomberg_markets', 'bloomberg_economics', 'bloomberg_tech', 'seekingalpha_all', 'seekingalpha_currents', 'fool', 'nasdaq_markets', 'investing_news', 'investing_economy', 'marketbeat', 'thestreet', 'kiplinger', 'fortune', 'business_insider', 'foxbusiness_latest', 'foxbusiness_markets', 'foxbusiness_economy', 'cbs_moneywatch', 'nypost_business', 'forbes_business', 'forbes_innovation', 'axios', 'semafor', 'nyt_business', 'nyt_economy', 'nyt_tech', 'nyt_home', 'latimes_business', 'guar

## 5. 실제 수집용 함수

통과 피드(`PASSED_FEEDS`)로 원하는 기간의 기사 목록을 모읍니다. 같은 언론사의 여러 섹션 피드에 같은 기사가 있으면
`(source, link)` 기준으로 합치고 섹션은 `sections`에 모아둡니다.

주의할 점
- `DATE_ARCHIVE` 피드는 4단계 깊이 결과에서 확인된 날짜까지는 같은 방식으로 수집 가능
- `PAGED` 피드는 마지막 페이지보다 과거로는 못 감
- 반환되는 것은 기사 목록(제목, 링크, 시각, 요약)이고 본문은 별도로 크롤링해야 함

In [14]:
def collect_news(start, end, feeds=None):
    feeds = PASSED_FEEDS if feeds is None else feeds
    articles = []
    stats = {}

    for feed_id, info in feeds.items():
        try:
            rows, stat = collect_feed(feed_id, info, start, end)
        except Exception as e:
            rows, stat = [], {"error": f"{type(e).__name__}: {e}"}

        articles.extend(rows)
        stats[feed_id] = {"articles": len(rows), **stat}

    df = pd.DataFrame(articles)

    if df.empty:
        return df, stats

    df = (
        df.sort_values("published_utc")
        .groupby(["source", "link"], as_index=False)
        .agg(
            name=("name", "first"),
            sections=("section", lambda x: ", ".join(sorted(set(x)))),
            feed_ids=("feed_id", lambda x: ", ".join(sorted(set(x)))),
            method=("method", "first"),
            title=("title", "first"),
            published_utc=("published_utc", "first"),
            published_et=("published_et", "first"),
            summary=("summary", "first"),
            author=("author", "first"),
        )
        .sort_values("published_utc")
        .reset_index(drop=True)
    )

    return df, stats


# 예시: 테스트 기간 첫날 하루 (3단계 캐시를 재사용하므로 대부분 네트워크 요청 없음)
news_df, news_stats = collect_news(TEST_START, TEST_START)

print(f"{TEST_START} 수집 기사 {len(news_df)}건 (언론사 {news_df['source'].nunique()}곳)")
news_df.groupby("name").size().sort_values(ascending=False).to_frame("articles")

2025-10-14 수집 기사 656건 (언론사 12곳)


,articles
name,
New York Post,246
The Hill,156
Fast Company,67
Investor's Business Daily,63
TechCrunch,28
9to5Mac,27
National Review,19
Semiconductor Engineering,15
WIRED,13


## 6. 눈으로 검수하기

피드별 표본 기사와 시간대별 기사 수 분포입니다. 특정 시간대가 통째로 비어 있으면 수집 누락을 의심하세요.

In [15]:
(
    articles_df
    .sort_values("published_utc")
    .groupby("feed_id")
    .head(2)[["feed_id", "method", "published_et", "title", "link"]]
)

,feed_id,method,published_et,title,link
376,nypost,DATE_ARCHIVE,2025-10-14 00:06:54-04:00,Mamdani’s biggest backer Democratic Socialists...,https://nypost.com/2025/10/14/us-news/zohran-m...
375,nypost,DATE_ARCHIVE,2025-10-14 00:28:43-04:00,"Cincinnati woman, 23, shot dead by friend duri...",https://nypost.com/2025/10/14/us-news/cincinna...
1203,9to5mac,DATE_ARCHIVE,2025-10-14 00:28:59-04:00,macOS 26.1 beta 3 changes polarizing Macintosh...,https://9to5mac.com/2025/10/13/macos-26-1-beta...
742,fastcompany,PAGED,2025-10-14 02:00:00-04:00,Most employers offer mental health care benefi...,https://www.fastcompany.com/91421154/most-empl...
50,ibd,DATE_ARCHIVE,2025-10-14 03:00:00-04:00,Cimpress Trying To Close In On Key Technical B...,https://www.investors.com/ibd-data-stories/cim...
45,ibd,DATE_ARCHIVE,2025-10-14 03:00:00-04:00,Brookfield Renewable Ptrs Getting Closer To Ke...,https://www.investors.com/ibd-data-stories/bro...
1270,eetimes,PAGED,2025-10-14 03:00:00-04:00,Diraq CEO Aims for Company’s First Quantum Com...,https://www.eetimes.com/diraq-ceo-aims-for-com...
1285,semiengineering,DATE_ARCHIVE,2025-10-14 03:01:26-04:00,Enabling In-Line Process Control for Hybrid Bo...,https://semiengineering.com/enabling-in-line-p...
1284,semiengineering,DATE_ARCHIVE,2025-10-14 03:01:30-04:00,Rethinking Security In Semiconductor Testing: ...,https://semiengineering.com/rethinking-securit...
1160,arstechnica,DATE_ARCHIVE,2025-10-14 03:05:16-04:00,SpaceX finally got exactly what it needed from...,https://arstechnica.com/space/2025/10/after-ye...


In [16]:
# 피드별 동부시간 시각대 기사 수 (테스트 기간 합계)
(
    articles_df
    .assign(hour_et=lambda d: d["published_et"].map(lambda t: t.hour))
    .pivot_table(index="feed_id", columns="hour_et", values="link", aggfunc="count", fill_value=0)
)

hour_et,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
feed_id,,,,,,,,,,,,,,,,,,,,,
9to5mac,1,0,0,0,0,0,1,4,5,9,...,6,2,7,3,2,2,1,0,2,1
arstechnica,0,0,0,1,0,0,0,0,0,4,...,3,2,4,3,0,0,0,0,0,0
eetimes,0,0,0,3,0,0,0,0,1,2,...,0,0,1,0,0,0,0,0,0,0
fastcompany,1,0,3,0,5,4,9,37,3,4,...,7,3,2,0,6,3,2,0,0,0
ibd,0,0,0,86,0,0,1,0,10,3,...,7,0,2,1,1,2,0,0,0,0
motherjones,0,0,0,0,0,0,2,1,0,0,...,2,0,0,4,1,0,0,0,0,0
nationalreview,1,1,0,0,0,0,7,1,4,4,...,4,3,4,3,2,1,1,0,0,0
nypost,5,10,5,6,0,3,14,18,11,20,...,21,35,49,35,34,19,26,23,15,9
semiengineering,0,0,0,18,0,0,0,0,0,0,...,2,0,0,1,1,0,0,0,0,0
